# BÀI TẬP: TITANIC
**Nguồn:** kaggle.com/c/titanic (891 dòng)


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

(891, 15)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object


## A.2. Missing values & Duplicate data

In [3]:
print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
Duplicates: 107


## A.3. Invalid values

In [4]:
print('Age <= 0:', (df.age <= 0).sum())
print('Fare < 0:', (df.fare < 0).sum())
print('Survived:', df.survived.unique())
print('Pclass:', df.pclass.unique())

Age <= 0: 0
Fare < 0: 0
Survived: [0 1]
Pclass: [3 1 2]


## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [5]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df[['sibsp', 'parch', 'family_size']].head()

,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
print(df[['age', 'fare', 'sibsp', 'parch', 'family_size']].mean())
print(df[['age', 'fare', 'sibsp', 'parch', 'family_size']].median())
print(df[['age', 'fare', 'sibsp', 'parch', 'family_size']].mode().iloc[0])

age            29.699118
fare           32.204208
sibsp           0.523008
parch           0.381594
family_size     1.904602
dtype: float64
age            28.0000
fare           14.4542
sibsp           0.0000
parch           0.0000
family_size     1.0000
dtype: float64
age            24.00
fare            8.05
sibsp           0.00
parch           0.00
family_size     1.00
Name: 0, dtype: float64


## Group 2 — Dispersion

In [7]:
print(df[['age', 'fare', 'family_size']].std())
print(df[['age', 'fare', 'family_size']].min())
print(df[['age', 'fare', 'family_size']].max())

age            14.526497
fare           49.693429
family_size     1.613459
dtype: float64
age            0.42
fare           0.00
family_size    1.00
dtype: float64
age             80.0000
fare           512.3292
family_size     11.0000
dtype: float64


## Group 3 — Location and Shape

In [8]:
print(df[['age', 'fare', 'family_size']].quantile([0.25, 0.5, 0.75]))
print(df[['age', 'fare', 'family_size']].skew())
print(df[['age', 'fare', 'family_size']].kurtosis())

         age     fare  family_size
0.25  20.125   7.9104          1.0
0.50  28.000  14.4542          1.0
0.75  38.000  31.0000          2.0
age            0.389108
fare           4.787317
family_size    2.727441
dtype: float64
age             0.178274
fare           33.398141
family_size     9.159666
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [9]:
result = df.groupby('pclass')['survived'].mean().sort_values(ascending=False)
print(result)
print('Difference:', result.iloc[0] - result.iloc[-1])

pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64
Difference: 0.3872671041713812


## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [10]:
print(df.groupby('sex')['survived'].mean())
print(df.groupby('pclass')['survived'].mean())

sex
female    0.742038
male      0.188908
Name: survived, dtype: float64
pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64


## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [11]:
print(df.groupby(pd.qcut(df['fare'], 4))['survived'].mean())
print(df[['fare', 'survived']].corr().iloc[0, 1])

fare
(-0.001, 7.91]     0.197309
(7.91, 14.454]     0.303571
(14.454, 31.0]     0.454955
(31.0, 512.329]    0.581081
Name: survived, dtype: float64
0.2573065223849626


## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [12]:
df['family_group'] = pd.cut(df['family_size'], [-1, 1, 4, np.inf], labels=['Alone', 'Small', 'Large'])
print(df.groupby('family_group', observed=True)['survived'].mean())

family_group
Alone    0.303538
Small    0.578767
Large    0.161290
Name: survived, dtype: float64


## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [13]:
print(df.groupby('embark_town', dropna=False)['survived'].mean().sort_values(ascending=False))

embark_town
NaN            1.000000
Cherbourg      0.553571
Queenstown     0.389610
Southampton    0.336957
Name: survived, dtype: float64


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

Dữ liệu Titanic có sự khác biệt rõ về tỷ lệ sống sót giữa các hạng vé. Giới tính và hạng vé đều liên quan đến khả năng sống sót, trong đó giới tính thường tạo ra chênh lệch lớn hơn. Hành khách trả vé cao hơn có xu hướng sống sót cao hơn. Nhóm gia đình nhỏ thường có tỷ lệ sống sót tốt hơn nhóm đi một mình hoặc gia đình quá đông. Tỷ lệ sống sót cũng thay đổi theo cảng lên tàu.